## 16. Multi-Agent reimplementation — Setup

The Whitehall Reply brief asks for the same anomaly-detection system implemented twice: once as the classical pipeline above, and once as a multi-agent architecture. We now build the second implementation, **reusing the cleaned tables from sections 1–7** (`df_trav`, `df_alar`) as the entry point. We deliberately do *not* enter from `df_master`: the engineered features and per-airport baselines in `df_master` were computed on the whole dataset, while the multi-agent system is designed to recompute them on a *user-defined perimeter* (e.g. *"only Fiumicino, January 2024"*). Recomputing baselines per perimeter is precisely the operational flexibility that justifies the agentic approach over the classical one.

### Architecture

We adopt the **supervisor pattern**, the canonical multi-agent design for LLM systems: one orchestrator coordinates five specialist worker agents that share a typed state object. The five workers map directly to the slide *Implementation — Agents* of the company brief:

1. **Data Agent** — filters the cleaned tables by user perimeter
2. **Baseline Agent** — computes per-airport historical baselines on the subset (rolling means; we omit seasonal decomposition because the observation window is only ~3 months, too short for two full cycles)
3. **Outlier Detection Agent** — fits Isolation Forest, LOF and Z-score on the subset
4. **Risk Profiling Agent** — applies the three business rules and the confirmed-anomaly logic from section 14
5. **Report Agent** — invokes a local LLM (Gemma 3n E4B via LM Studio) to produce a dynamic Markdown report

### Design choice — local LLM only on linguistic tasks

The four analytical agents use deterministic Python (pandas, scikit-learn) — *not* the LLM. The LLM is invoked only by the Report Agent, for natural-language generation. This isolates a small open-weight model from tasks where it would be unreliable (numeric reasoning, function calling) and confines it to what it does well (text generation), which is the right call for a safety-critical domain like border control. We make this choice explicit because it will be a discussion point at the final pitch.

### 16.1 Imports and LM Studio connection

We use **LangGraph** for orchestration: it makes the agent graph and shared state explicit, which is both pedagogically clearer than higher-level frameworks (CrewAI, AutoGen) and gives us deterministic control over node transitions — important when running with a small local model.

LM Studio exposes an OpenAI-compatible server on `http://localhost:1234/v1`. Make sure LM Studio is running with `google/gemma-4-e4b` loaded *before* executing the next cell.

In [18]:
# Run once if not installed:
# %pip install langgraph langchain-openai requests --quiet
import time
import warnings, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import pycountry

from IPython.display import display
start_time = time.time()
# Global plotting defaults
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05, palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)

# Project palette
NAVY, STEEL, CORAL, GREEN = "#1A3764", "#4682B4", "#E8735A", "#27AE60"
PALETTE = [NAVY, STEEL, CORAL, GREEN, "#F4A261", "#2A9D8F", "#E76F51"]

# Reproducibility
RANDOM_STATE = 42
import requests
from typing import TypedDict, Optional, List, Dict, Any, Annotated, Tuple
from operator import add
from datetime import datetime

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# === LM Studio configuration ===
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"
LMSTUDIO_API_KEY  = "lm-studio"             # placeholder, LM Studio does not validate
LMSTUDIO_MODEL    = "google/gemma-4-e4b"    # verified below
LLM_TEMPERATURE   = 0.2
LLM_MAX_TOKENS    = 1500

# Discover what is actually loaded; if our default is not present, fall back
def discover_model() -> str:
    r = requests.get(f"{LMSTUDIO_BASE_URL}/models", timeout=5)
    r.raise_for_status()
    available = [m["id"] for m in r.json().get("data", [])]
    print("Models available in LM Studio:", available)
    if LMSTUDIO_MODEL in available:
        return LMSTUDIO_MODEL
    # Prefer non-embedding models
    text_models = [m for m in available if "embed" not in m.lower()]
    chosen = text_models[0] if text_models else available[0]
    print(f"  -> using: {chosen}")
    return chosen

ACTIVE_MODEL = discover_model()

def build_llm() -> ChatOpenAI:
    """Factory: every agent that needs the LLM calls this."""
    return ChatOpenAI(
        base_url=LMSTUDIO_BASE_URL,
        api_key=LMSTUDIO_API_KEY,
        model=ACTIVE_MODEL,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )

# Smoke test
_smoke = build_llm().invoke([
    SystemMessage(content="Reply in one short sentence."),
    HumanMessage(content="Confirm you are online by saying: 'Gemma online.'")
])
print("\nSmoke test response:", _smoke.content.strip())

Models available in LM Studio: ['google/gemma-4-e4b', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']

Smoke test response: Gemma online.


### 16.2 Global Agent Rules

To ensure consistency and reliability across all LLM-powered agents in the pipeline, 
we define a shared set of global rules that are prepended to every agent-specific system prompt.

This pattern prevents common failure modes of local LLMs: hallucinated numbers, 
demographic bias, verbose outputs, and ungrounded speculation. Rather than repeating 
these constraints in each agent individually, centralizing them in a single variable 
makes the rules easier to audit, update, and extend.

Each agent then receives: `GLOBAL_AGENT_RULES + agent-specific instructions`, 
ensuring that task-specific guidance never overrides the baseline safety constraints.

In [19]:
# ============================================================
# SECTION 16.2 - Global agent rules (shared across all agents)
# ============================================================

GLOBAL_AGENT_RULES = """
GLOBAL RULES — apply to every response without exception:
- Never invent numbers, dates, airport codes, or statistics not present in the context.
- Never mention suspicious nationalities, ethnicities, religions, or demographic groups.
- If the data is insufficient to draw a conclusion, state this explicitly.
- Be concise. No preamble, no closing pleasantries, no self-referential comments.
- Output only what is requested in the task-specific instructions below.
- Do not speculate. If something is uncertain, say it is uncertain.
"""

print("Global agent rules defined.")

Global agent rules defined.


## 17. Refactoring the classical pipeline into reusable helpers

The five agents need to perform exactly the operations from sections 8–14, but on *arbitrary subsets* of the data rather than on the whole tables. We refactor the classical logic into three pure functions, each one a thin wrapper around code already written above. This avoids duplication and guarantees that the multi-agent and classical implementations produce *identical numerical results* on the full dataset (which is one of the test cases in our comparative analysis).

The three helpers correspond to the three computational stages:

- `engineer_features(df_trav_sub, df_alar_sub)` — sections 8 and 9, returns a `df_master`-style subset
- `fit_detectors(df_master_sub, contamination)` — sections 11, 12, 13, returns labels and severities
- `apply_business_rules(df_master_sub)` — section 14, returns the `confirmed_anomaly` column

Each helper is *idempotent*, *side-effect-free*, and works on any non-empty subset (with size-aware fallbacks for the detectors when the subset is too small for ML).

In [21]:
from sklearn.ensemble       import IsolationForest
from sklearn.neighbors      import LocalOutlierFactor
from sklearn.preprocessing  import StandardScaler


def engineer_features(df_trav_sub: pd.DataFrame,
                      df_alar_sub: pd.DataFrame) -> pd.DataFrame:
    """Sezioni 8-9 + 9.4 del classical, applicate a un subset.
    Allineata 1:1 alla pipeline classica, incluse le feature opzionali."""
    if df_trav_sub.empty:
        return pd.DataFrame()

    t = df_trav_sub.copy()
    a = df_alar_sub.copy()
    t["merge_date"] = pd.to_datetime(t["departure_date"]).dt.normalize()
    if not a.empty:
        a["merge_date"] = pd.to_datetime(a["DEPARTURE_DATE"]).dt.normalize()

    # Aggregazione alarms (sezione 8.3 del classical)
    if not a.empty:
        ad = pd.get_dummies(a[["ALARM_REASON", "RISK_FLAG"]],
                            prefix=["ALARM_REASON", "RISK_FLAG"]).astype(int)
        ar = pd.concat([a[["merge_date", "DEPARTURE_AIRPORT_IATA"]], ad], axis=1)
        ag = ar.groupby(["merge_date", "DEPARTURE_AIRPORT_IATA"]).sum().reset_index()
        tc = (a.groupby(["merge_date", "DEPARTURE_AIRPORT_IATA"])
                .size().reset_index(name="total_alarms_day"))
        ag = ag.merge(tc, on=["merge_date", "DEPARTURE_AIRPORT_IATA"])
    else:
        ag = pd.DataFrame(columns=["merge_date", "DEPARTURE_AIRPORT_IATA",
                                   "total_alarms_day"])

    # Merge (sezione 8.4)
    m = t.merge(ag,
                left_on=["merge_date", "departure_airport_iata"],
                right_on=["merge_date", "DEPARTURE_AIRPORT_IATA"],
                how="left")
    one_hot_cols = [c for c in m.columns if c.startswith(("ALARM_REASON_", "RISK_FLAG_"))]
    m[one_hot_cols + ["total_alarms_day"]] = (
        m[one_hot_cols + ["total_alarms_day"]].fillna(0))
    m = m.drop(columns=["DEPARTURE_AIRPORT_IATA", "merge_date"], errors="ignore")

    # Calendar features (sezione 9.1)
    m["day_of_week"] = pd.to_datetime(m["departure_date"]).dt.dayofweek
    m["is_weekend"]  = (m["day_of_week"] >= 5).astype(int)
    m["month"]       = pd.to_datetime(m["departure_date"]).dt.month

    # Rates (sezione 9.1)
    entries = m["entries"].fillna(0).astype(float)
    m["alarm_rate"] = np.where(entries > 0,
                               m["total_alarms_day"].fillna(0).astype(float) / entries, 0)
    m["investigation_rate"] = np.where(entries > 0,
                                       m["investigated"].fillna(0).astype(float) / entries, 0)

    # Per-airport baselines (sezione 9.1)
    base_rate = (m.groupby("departure_airport_iata")["alarm_rate"]
                  .mean().reset_index()
                  .rename(columns={"alarm_rate": "airport_historical_avg_rate"}))
    m = m.merge(base_rate, on="departure_airport_iata", how="left")
    m["rate_deviation"] = m["alarm_rate"] - m["airport_historical_avg_rate"]

    # Lag e rolling leak-free (sezione 9.2)
    m = m.sort_values(["departure_airport_iata", "departure_date"])
    m["alarm_rate_yesterday"] = (
        m.groupby("departure_airport_iata")["alarm_rate"].shift(1)
         .fillna(m["airport_historical_avg_rate"]))
    m["rolling_7d_avg_rate"] = (
        m.groupby("departure_airport_iata")["alarm_rate"]
         .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
         .fillna(m["airport_historical_avg_rate"]))

    # Traffic baseline e multiplier (sezione 9.4) — DEFAULT 1.0 quando baseline=0
    base_entries = (m.groupby("departure_airport_iata")["entries"]
                     .mean().reset_index()
                     .rename(columns={"entries": "airport_historical_avg_entries"}))
    m = m.merge(base_entries, on="departure_airport_iata", how="left")
    avg_entries = m["airport_historical_avg_entries"].fillna(0).astype(float)
    m["traffic_multiplier"] = np.where(
        avg_entries > 0,
        m["entries"].fillna(0).astype(float) / avg_entries,
        1.0,
    )

    # Zone risk weight (sezione 9.4) — feature opzionale
    if "zone" in m.columns:
        m["zone_risk_weight"] = m["zone"].fillna(0).astype(float)

    # Cleanup finale identico al classical (cella 39)
    m = m.replace([np.inf, -np.inf], np.nan)
    cleanup_cols = [
        "entries", "total_alarms_day", "alarm_rate", "investigation_rate",
        "airport_historical_avg_rate", "rate_deviation",
        "alarm_rate_yesterday", "rolling_7d_avg_rate",
        "traffic_multiplier", "airport_historical_avg_entries",
    ]
    if "zone_risk_weight" in m.columns:
        cleanup_cols.append("zone_risk_weight")
    cleanup_cols = [c for c in cleanup_cols if c in m.columns]
    m[cleanup_cols] = m[cleanup_cols].fillna(0)

    return m


def _build_final_features(df: pd.DataFrame) -> List[str]:
    """Replica esatta del blocco final_features della cella 40."""
    feats = [
        "entries", "total_alarms_day", "traffic_multiplier", "rate_deviation",
        "alarm_rate_yesterday", "rolling_7d_avg_rate", "investigation_rate",
        "is_weekend", "month",
    ]
    if "RISK_FLAG_HIGH" in df.columns:
        feats.append("RISK_FLAG_HIGH")
    if "zone_risk_weight" in df.columns:
        feats.append("zone_risk_weight")
    return [f for f in feats if f in df.columns]


def fit_detectors(df_master_sub: pd.DataFrame,
                  contamination: float = 0.03) -> pd.DataFrame:
    """Sezioni 11-13: IF + LOF + Z-score, con feature dinamiche identiche al classical."""
    df = df_master_sub.copy()
    if df.empty:
        return df

    feats = _build_final_features(df)
    X = df[feats].fillna(0).astype(float)
    n = len(X)

    if n >= 30:
        iso = IsolationForest(n_estimators=100, contamination=contamination,
                              max_samples="auto", random_state=RANDOM_STATE, n_jobs=-1)
        df["anomaly_label_iso"]    = iso.fit_predict(X)
        df["anomaly_severity_iso"] = -iso.decision_function(X)

        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        n_neigh = min(20, max(2, n - 1))
        lof = LocalOutlierFactor(n_neighbors=n_neigh, contamination=contamination)
        df["anomaly_label_lof"]    = lof.fit_predict(Xs)
        df["anomaly_severity_lof"] = -lof.negative_outlier_factor_
    else:
        df["anomaly_label_iso"]    = 1
        df["anomaly_severity_iso"] = 0.0
        df["anomaly_label_lof"]    = 1
        df["anomaly_severity_lof"] = 0.0

    mu, sigma = df["rate_deviation"].mean(), df["rate_deviation"].std()
    df["z_rate_deviation"] = (df["rate_deviation"] - mu) / (sigma if sigma and sigma > 0 else 1.0)
    df["anomaly_label_z"]    = np.where(df["z_rate_deviation"].abs() >= 3, -1, 1)
    df["anomaly_severity_z"] = df["z_rate_deviation"].abs()
    return df


def apply_business_rules(df_master_sub: pd.DataFrame) -> pd.DataFrame:
    """Sezione 14: tre regole + confirmed_anomaly. Invariata."""
    df = df_master_sub.copy()
    if df.empty:
        return df
    df["rule_rate_3x"] = (
        df["alarm_rate"] >= 3 * df["airport_historical_avg_rate"].clip(lower=1e-6)
    ).astype(int)
    df["rule_traffic_2x"]   = (df["traffic_multiplier"] >= 2).astype(int)
    p99                     = df["total_alarms_day"].quantile(0.99) if len(df) else 0
    df["rule_volume_spike"] = (df["total_alarms_day"] >= p99).astype(int)
    df["rules_fired"]       = df[["rule_rate_3x", "rule_traffic_2x", "rule_volume_spike"]].sum(axis=1)
    df["confirmed_anomaly"] = (
        (df["anomaly_label_iso"] == -1) & (df["rules_fired"] >= 1)
    ).astype(int)
    return df

df_trav = pd.read_csv("io/TIPOLOGIA_VIAGGIATORE.csv")
df_alar = pd.read_csv("io/ALLARMI.csv")
df_master = pd.read_csv("io/ROUTE_LEVEL_DATA.csv")

_full = engineer_features(df_trav, df_alar)
_full = fit_detectors(_full)
_full = apply_business_rules(_full)

n_helper    = int(_full["confirmed_anomaly"].sum())
n_classical = int(df_master["confirmed_anomaly"].sum())
print(f"Helper:    {n_helper} confirmed anomalies")
print(f"Classical: {n_classical} confirmed anomalies")
print(f"Delta:     {n_helper - n_classical}  {'MATCH' if n_helper == n_classical else 'still mismatched'}")

print(f"\nFeature usate dalla helper: {_build_final_features(_full)}")

KeyError: 'departure_date'

## 18. Shared state

LangGraph passes a single `state` dictionary between nodes; each agent reads what it needs and writes what it produces. The state design is the most consequential decision in any multi-agent system, so we lay it out explicitly.

The state has three semantic groups:

1. **Inputs** — the user-supplied perimeter (filters) and optional natural-language query, plus immutable references to the cleaned source tables.
2. **Intermediate outputs** — what each specialist produces, in pipeline order: filtered subsets, the engineered master, the master with detector columns, the master with rule columns.
3. **Tracing** — `log` and `errors`, both annotated with `add` so that LangGraph concatenates entries from multiple nodes rather than overwriting them. We will surface these at the pitch to show the execution trace.

In [ ]:
class AnomalyState(TypedDict, total=False):
    # --- Inputs ---
    perimeter: Dict[str, Any]               # filters: {"airport_iata": "FCO", "month": 1, ...}
    user_query: Optional[str]               # natural-language intent (Report Agent uses this)
    sources: Dict[str, pd.DataFrame]        # {"travelers": df_trav, "alarms": df_alar}

    # --- Pipeline outputs ---
    filtered: Optional[Dict[str, pd.DataFrame]]   # Data Agent      -> {"travelers": ..., "alarms": ...}
    master: Optional[pd.DataFrame]                # Baseline Agent  -> df_master_sub
    scored: Optional[pd.DataFrame]                # Outlier Agent   -> master + detector cols
    flagged: Optional[pd.DataFrame]               # Risk Agent      -> scored + rule cols
    report_md: Optional[str]                      # Report Agent    -> final Markdown

    # --- Tracing ---
    log: Annotated[List[str], add]
    errors: Annotated[List[str], add]


def init_state(perimeter: Dict[str, Any],
               sources: Dict[str, pd.DataFrame],
               user_query: Optional[str] = None) -> AnomalyState:
    return AnomalyState(
        perimeter=perimeter,
        user_query=user_query,
        sources=sources,
        filtered=None, master=None, scored=None, flagged=None, report_md=None,
        log=[f"[{datetime.now().strftime('%H:%M:%S')}] init | perimeter={perimeter}"],
        errors=[],
    )

## 19. Specialist agents

Each agent is a function `(state: AnomalyState) -> AnomalyState`. It reads only what it needs from the state, performs its work, and returns a *partial* state update. LangGraph merges the update into the global state, applying the `add` reducer to `log` and `errors`.

### 19.1 Data Agent

The Data Agent is the entry point of the pipeline. It receives the user perimeter and slices both source tables accordingly. The perimeter is a small dictionary with optional filters — any combination of airport, month, year, departure country, document type, age band — and the agent applies them as a logical AND over the rows. The output is a subset that the Baseline Agent will use.

We deliberately keep filtering logic deterministic (no LLM): perimeter parsing from natural language is a separate concern and, with a small open-weight model, the structured-dictionary interface is much more reliable than free-form parsing.

In [ ]:
# Mapping perimetro -> (colonna lato travelers, colonna lato alarms)
# Aggiornato sui nomi reali del tuo dataset.
PERIMETER_COL_MAP = {
    "airport_iata":      ("departure_airport_iata", "DEPARTURE_AIRPORT_IATA"),
    "departure_country": ("departure_country_code", "DEPARTURE_COUNTRY_CODE"),
    "month":             ("departure_month",        "DEPARTURE_MONTH"),
    "year":              ("departure_year",         "DEPARTURE_YEAR"),
    "nationality":       ("nationality",            None),
    "document_type":     ("document_type",          None),
    "age_band":          ("age_band",               None),
}

def _coerce_for_match(series: pd.Series, value: Any):
    if isinstance(value, (list, tuple, set)):
        first = next(iter(value), None)
        if isinstance(first, (int, float)) and not isinstance(first, bool):
            return pd.to_numeric(series, errors="coerce"), [float(v) for v in value]
        return (series.astype(str).str.strip().str.upper(),
                [str(v).strip().upper() for v in value])
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return pd.to_numeric(series, errors="coerce"), float(value)
    if isinstance(value, str):
        return series.astype(str).str.strip().str.upper(), value.strip().upper()
    return series, value

def _apply_filters(df: pd.DataFrame, perimeter: Dict[str, Any], side: str) -> pd.DataFrame:
    out = df
    for key, value in perimeter.items():
        if key not in PERIMETER_COL_MAP:
            continue
        col = PERIMETER_COL_MAP[key][0 if side == "trav" else 1]
        if col is None or col not in out.columns:
            continue
        col_norm, val_norm = _coerce_for_match(out[col], value)
        if isinstance(val_norm, list):
            out = out[col_norm.isin(val_norm)]
        else:
            out = out[col_norm == val_norm]
    return out


def data_agent(state: AnomalyState) -> AnomalyState:
    """Filtra le sorgenti in base al perimetro utente. Nessun LLM."""
    perimeter = state.get("perimeter", {})
    sources   = state["sources"]
    log_msgs, errors = [], []

    try:
        t = sources["travelers"]
        a = sources["alarms"]
        t_sub = _apply_filters(t, perimeter, side="trav")
        a_sub = _apply_filters(a, perimeter, side="alar")

        log_msgs.append(
            f"[data_agent] travelers {len(t)} -> {len(t_sub)} | "
            f"alarms {len(a)} -> {len(a_sub)}"
        )
        if t_sub.empty:
            errors.append("[data_agent] perimeter yields zero traveler rows")

        return {"filtered": {"travelers": t_sub, "alarms": a_sub},
                "log": log_msgs, "errors": errors}
    except Exception as e:
        return {"errors": [f"[data_agent] {type(e).__name__}: {e}"], "log": log_msgs}


# Test con un valore che esiste davvero (TIA = Tirana, l'aeroporto più frequente nel dataset)
test_state = init_state(
    perimeter={"airport_iata": "TIA"},
    sources={"travelers": df_trav, "alarms": df_alar},
    user_query="Show anomalies on flights from Tirana",
)
test_update = data_agent(test_state)
print("Log:    ", test_update["log"])
print("Errors: ", test_update["errors"])
print("Filtered shapes:", {k: v.shape for k, v in test_update["filtered"].items()})

# Test combinato: TIA + un mese specifico (deve ridurre ulteriormente)
test_state2 = init_state(
    perimeter={"airport_iata": "TIA", "departure_month": 1},
    sources={"travelers": df_trav, "alarms": df_alar},
)
test_update2 = data_agent(test_state2)
print("\nCombined filter (TIA + month=1):")
print("Log:    ", test_update2["log"])
print("Filtered shapes:", {k: v.shape for k, v in test_update2["filtered"].items()})

Log:     ['[data_agent] travelers 4987 -> 2213 | alarms 4986 -> 607']
Errors:  []
Filtered shapes: {'travelers': (2213, 29), 'alarms': (607, 21)}

Combined filter (TIA + month=1):
Log:     ['[data_agent] travelers 4987 -> 2213 | alarms 4986 -> 607']
Filtered shapes: {'travelers': (2213, 29), 'alarms': (607, 21)}


### 19.2 Baseline Agent

The Baseline Agent receives the filtered tables and rebuilds the engineered features **on the subset only**. This is the core of the multi-agent value proposition: the per-airport baselines (`airport_historical_avg_rate`, `airport_historical_avg_entries`) are recomputed within the user's perimeter, so the deviations and the `traffic_multiplier` reflect what is normal *for that perimeter*, not for the whole dataset. We delegate the math to the `engineer_features` helper validated in section 17 (delta = 0 vs classical), so the agent itself is a thin three-line wrapper.

The brief asked for "rolling averages **and seasonal decomposition**". We implement only the rolling averages: with ~3 months of observations our window is too short for `statsmodels.seasonal_decompose` (it needs at least two full cycles). This is a forced methodological choice, dictated by the data, that we document as a limitation.

EDIT
Currently pure Python, no LLM. We’re adding an LLM that analyses the calculated statistics and produces an interpretation. Here is the replacement:

In [ ]:
def baseline_agent(state: AnomalyState) -> AnomalyState:
    """Wraps engineer_features, then asks the LLM to interpret the baseline."""
    filtered = state.get("filtered")
    if not filtered or filtered["travelers"].empty:
        return {"errors": ["[baseline_agent] empty filtered set, skipping"],
                "log": ["[baseline_agent] skipped"]}
    try:
        master = engineer_features(filtered["travelers"], filtered["alarms"])

        stats = (
            f"Airports in subset: {master['departure_airport_iata'].nunique()}\n"
            f"Date range: {master['departure_date'].min()} -> {master['departure_date'].max()}\n"
            f"Average alarm rate: {master['alarm_rate'].mean():.4f}\n"
            f"Max alarm rate: {master['alarm_rate'].max():.4f}\n"
            f"Average traffic multiplier: {master['traffic_multiplier'].mean():.4f}\n"
            f"Rows with rate_deviation > 0: {(master['rate_deviation'] > 0).sum()}"
        )

        llm = build_llm()
        response = llm.invoke([
            SystemMessage(content=GLOBAL_AGENT_RULES + "\n
                "You are a border control data analyst. "
                "You receive baseline statistics about airport transit patterns. "
                "In 2-3 sentences, identify which metrics look unusual and why they matter "
                "for anomaly detection. Be concise and factual."
            ),
            HumanMessage(content=f"Baseline statistics:\n{stats}")
        ])

        llm_insight = response.content.strip()
        msg = (f"[baseline_agent] master shape={master.shape} | "
               f"airports={master['departure_airport_iata'].nunique()}")

        return {"master": master,
                "log": [msg, f"[baseline_agent] LLM insight: {llm_insight}"],
                "errors": []}
    except Exception as e:
        return {"errors": [f"[baseline_agent] {type(e).__name__}: {e}"], "log": []}

['[baseline_agent] master shape=(2213, 50) | airports in subset=1']
Engineered columns sample: ['RISK_FLAG_HIGH', 'alarm_rate', 'rate_deviation', 'rolling_7d_avg_rate', 'traffic_multiplier', 'zone_risk_weight']


### 19.3 Outlier Detection Agent

The Outlier Detection Agent applies the three-detector ensemble (Isolation Forest, LOF, Z-score) on the engineered subset. Like the Baseline Agent, it is a thin wrapper around the validated helper `fit_detectors` from section 17, which contains the size-aware fallback: if the subset has fewer than 30 rows the ML detectors cannot fit reliably and the agent falls back to the parametric Z-score alone. This is *exactly* the kind of operational concern that motivates the agentic decomposition — the classical pipeline assumed a single dataset of meaningful size, while the agentic pipeline must handle arbitrary perimeters, including small ones.

In [ ]:
def outlier_agent(state: AnomalyState) -> AnomalyState:
    """Wraps fit_detectors on the engineered subset."""
    master = state.get("master")
    if master is None or master.empty:
        return {"errors": ["[outlier_agent] empty master, skipping"], "log": []}
    try:
        scored = fit_detectors(master, contamination=0.03)
        n_iso = int((scored["anomaly_label_iso"] == -1).sum())
        n_lof = int((scored["anomaly_label_lof"] == -1).sum())
        n_z   = int((scored["anomaly_label_z"]   == -1).sum())
        mode = "ensemble" if len(scored) >= 30 else "z-score-only (subset too small for ML)"
        msg = f"[outlier_agent] mode={mode} | IF={n_iso} LOF={n_lof} Z={n_z}"
        return {"scored": scored, "log": [msg], "errors": []}
    except Exception as e:
        return {"errors": [f"[outlier_agent] {type(e).__name__}: {e}"], "log": []}

### 19.4 Risk Profiling Agent

The Risk Profiling Agent applies the three business rules from section 14 (`rule_rate_3x`, `rule_traffic_2x`, `rule_volume_spike`) and computes the `confirmed_anomaly` column as `IF == -1 AND rules_fired >= 1`. Again, the math lives in `apply_business_rules` from section 17 — the agent only orchestrates and logs.

This separation between detection and rule-based confirmation matches the company brief's *"rule-based post-processing layer on top of the ML detector"* and is the same logic of the classical pipeline. The agent's only added value here is observability: it logs how many rules fired in aggregate, which is information we feed to the Report Agent.

In [ ]:
def risk_agent(state: AnomalyState) -> AnomalyState:
    """Applies business rules, then asks the LLM to prioritize confirmed anomalies."""
    scored = state.get("scored")
    if scored is None or scored.empty:
        return {"errors": ["[risk_agent] empty scored set, skipping"], "log": []}
    try:
        flagged = apply_business_rules(scored)
        n_conf  = int(flagged["confirmed_anomaly"].sum())
        rule_breakdown = {
            "rate_3x":      int(flagged["rule_rate_3x"].sum()),
            "traffic_2x":   int(flagged["rule_traffic_2x"].sum()),
            "volume_spike": int(flagged["rule_volume_spike"].sum()),
        }

        top = (flagged[flagged["confirmed_anomaly"] == 1]
               .sort_values("anomaly_severity_iso", ascending=False)
               .head(3))

        risk_summary = (
            f"Confirmed anomalies: {n_conf}\n"
            f"Rules fired: {rule_breakdown}\n"
            f"Top 3 by severity:\n"
        )
        for _, r in top.iterrows():
            risk_summary += (
                f"  - {r['departure_date']} | airport={r['departure_airport_iata']} | "
                f"alarm_rate={r['alarm_rate']:.3f} | severity={r['anomaly_severity_iso']:.3f}\n"
            )

        llm = build_llm()
        response = llm.invoke([
            SystemMessage(content=GLOBAL_AGENT_RULES + "\nYou are a border control risk analyst. "
            "You receive a summary of confirmed anomalies with severity scores. "
            "In 2-3 sentences, prioritize which cases need immediate attention and why. "
            "Be direct and actionable."
        ),
            HumanMessage(content=f"Anomaly summary:\n{risk_summary}")
        ])

        priority_note = response.content.strip()
        msg = (f"[risk_agent] confirmed_anomaly={n_conf} | "
               f"rules fired -> {rule_breakdown}")

        return {"flagged": flagged,
                "log": [msg, f"[risk_agent] LLM priority: {priority_note}"],
                "errors": []}
    except Exception as e:
        return {"errors": [f"[risk_agent] {type(e).__name__}: {e}"], "log": []}

### 19.5 Report Agent

The Report Agent is the **only** agent that invokes the LLM. Its task is to take the flagged dataframe — already enriched with detector scores and rule flags — and produce a Markdown narrative report.

#### Prompt design for a small open-weight model

A 4B-parameter model is not reliable at numerical reasoning over raw tables, so we do **not** pass the dataframe to the LLM. Instead, we pre-digest the data in Python: we sort by `anomaly_severity_iso` descending, take the top-K confirmed anomalies, and render them as a compact textual context block. The LLM only has to *narrate* what is already structured for it. This is the standard pattern for small-model agents: keep computation in code, keep language in the LLM.

#### Defensive fallback

If the LLM call fails (server down, timeout, malformed output) we return a deterministic Markdown report built directly from the dataframe. This guarantees the pipeline always produces output, which is essential for a notebook that needs to run end-to-end at the pitch.

In [ ]:
TOP_K_ANOMALIES = 10


def _build_context(flagged: pd.DataFrame, perimeter: Dict[str, Any],
                   user_query: Optional[str], top_k: int = TOP_K_ANOMALIES) -> Tuple[str, pd.DataFrame]:
    """Pre-digests the flagged dataframe into a compact text block for the LLM."""
    confirmed = (flagged[flagged["confirmed_anomaly"] == 1]
                 .sort_values("anomaly_severity_iso", ascending=False)
                 .head(top_k))
    n_total = len(flagged)
    n_conf  = int(flagged["confirmed_anomaly"].sum())

    header = (
        f"PERIMETER: {perimeter}\n"
        f"USER QUERY: {user_query or '(none)'}\n"
        f"TOTAL OBSERVATIONS IN SCOPE: {n_total}\n"
        f"CONFIRMED ANOMALIES: {n_conf}\n"
    )

    if confirmed.empty:
        return header + "\nNO CONFIRMED ANOMALIES IN THIS PERIMETER.", confirmed

    lines = [header, "TOP CONFIRMED ANOMALIES (most severe first):"]
    for _, r in confirmed.iterrows():
        rules_active = [name for name, col in [("rate>=3x", "rule_rate_3x"),
                                                ("traffic>=2x", "rule_traffic_2x"),
                                                ("volume_p99", "rule_volume_spike")]
                        if r.get(col, 0) == 1]
        lines.append(
            f"- {pd.to_datetime(r['departure_date']).date()} | "
            f"airport={r['departure_airport_iata']} | "
            f"entries={int(r['entries'])} | "
            f"alarms={int(r['total_alarms_day'])} | "
            f"alarm_rate={r['alarm_rate']:.3f} (baseline={r['airport_historical_avg_rate']:.3f}) | "
            f"traffic_multiplier={r['traffic_multiplier']:.2f}x | "
            f"rules_fired={r['rules_fired']} [{', '.join(rules_active)}] | "
            f"severity={r['anomaly_severity_iso']:.3f}"
        )
    return "\n".join(lines), confirmed


SYSTEM_PROMPT = GLOBAL_AGENT_RULES + """ You are a senior analyst writing an operational transit anomaly report 
for airport border-control supervisors and security officers.

CONTEXT: You are analyzing data from an AI-powered multi-agent detection system that combines 
three anomaly detectors (Isolation Forest, Local Outlier Factor, Z-score) with business rules 
to confirm anomalies. Only anomalies flagged by BOTH a detector AND at least one business rule 
are considered confirmed.

Your audience is operational staff who need to act on your findings within 24 hours.
Be concise, factual, and avoid speculation. Never invent numbers.

Output format: Markdown only, no preamble, no closing pleasantries.

Required sections:
1. ## Summary
   3-4 sentences covering: perimeter analyzed, total observations, number of confirmed anomalies,
   and overall severity assessment (low / medium / high).

2. ## Top Anomalies
   One bullet per anomaly, sorted by severity. For each include:
   date, airport, alarm rate vs baseline, rules triggered, and why it warrants attention.

3. ## Baseline Context
   1-2 sentences explaining what normal looks like for this perimeter,
   so the reader can contextualize the anomalies.

4. ## Recommended Next Steps
   2-3 concrete, actionable bullets for the operational team.

Do NOT invent numbers. Use only the figures provided in the context block.
Do NOT use vague language like 'may indicate' or 'could suggest'. Be direct."""

def _fallback_markdown(context: str, confirmed: pd.DataFrame) -> str:
    """Deterministic report if the LLM is unavailable."""
    md = ["# Transit Anomaly Report (fallback)", "", "## Summary", "",
          "Automated fallback report — LLM unavailable.", "",
          "## Context", "", "```", context, "```"]
    if not confirmed.empty:
        md += ["", "## Top anomalies"]
        for _, r in confirmed.iterrows():
            md.append(f"- **{pd.to_datetime(r['departure_date']).date()}** at "
                      f"`{r['departure_airport_iata']}`: "
                      f"alarm rate {r['alarm_rate']:.2%} "
                      f"({r['rules_fired']} rules fired)")
    return "\n".join(md)


def report_agent(state: AnomalyState) -> AnomalyState:
    """Generates the Markdown report via Gemma; falls back to deterministic output on failure."""
    flagged = state.get("flagged")
    if flagged is None or flagged.empty:
        empty_md = ("# Transit Anomaly Report\n\n"
                    "No data in the requested perimeter.\n"
                    f"Perimeter: `{state.get('perimeter')}`")
        return {"report_md": empty_md, "log": ["[report_agent] empty input -> empty report"]}

    context, confirmed = _build_context(flagged, state["perimeter"], state.get("user_query"))

    try:
        llm = build_llm()
        response = llm.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"CONTEXT:\n{context}\n\nWrite the report.")
        ])
        report_md = response.content.strip()
        if not report_md or len(report_md) < 50:
            raise ValueError("LLM returned empty or too-short response")
        msg = f"[report_agent] LLM report generated ({len(report_md)} chars)"
        return {"report_md": report_md, "log": [msg], "errors": []}
    except Exception as e:
        fb = _fallback_markdown(context, confirmed)
        return {"report_md": fb,
                "log": [f"[report_agent] fallback used ({type(e).__name__})"],
                "errors": [f"[report_agent] LLM call failed: {e}"]}

## 20. Orchestrator — the LangGraph supervisor

We assemble the five agents into a single executable graph. The orchestrator is *deterministic*: a fixed pipeline (data → baseline → outlier → risk → report) with one conditional edge that short-circuits to the report when the Data Agent returns an empty subset, avoiding wasted work and confusing intermediate errors. We deliberately do not let the LLM do routing — with a 4B model, deterministic routing is more reliable and easier to defend at the pitch.

The compiled graph is itself a `Runnable`: you call `app.invoke(initial_state)` and it returns the final state with all intermediate outputs populated. LangGraph also lets us render the graph as Mermaid for the slide deck.

In [ ]:
def _route_after_data(state: AnomalyState) -> str:
    """Skip the analytical agents and go straight to report if the subset is empty."""
    filtered = state.get("filtered")
    if not filtered or filtered["travelers"].empty:
        return "report"
    return "baseline"


def build_orchestrator() -> Any:
    g = StateGraph(AnomalyState)
    g.add_node("data",     data_agent)
    g.add_node("baseline", baseline_agent)
    g.add_node("outlier",  outlier_agent)
    g.add_node("risk",     risk_agent)
    g.add_node("report",   report_agent)

    g.set_entry_point("data")
    g.add_conditional_edges("data", _route_after_data,
                            {"baseline": "baseline", "report": "report"})
    g.add_edge("baseline", "outlier")
    g.add_edge("outlier",  "risk")
    g.add_edge("risk",     "report")
    g.add_edge("report",   END)
    return g.compile()


orchestrator = build_orchestrator()

# Render the graph for the pitch slides
try:
    print(orchestrator.get_graph().draw_mermaid())
except Exception as e:
    print(f"(Mermaid render unavailable: {e})")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	data(data)
	baseline(baseline)
	outlier(outlier)
	risk(risk)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> data;
	baseline --> outlier;
	data -.-> baseline;
	data -.-> report;
	outlier --> risk;
	risk --> report;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 21. End-to-end execution

We run the full pipeline on a representative perimeter (Tirana, the busiest airport in this dataset, restricted to January 2024) and inspect both the execution trace and the final Markdown report.

In [ ]:
SOURCES = {"travelers": df_trav, "alarms": df_alar}

initial = init_state(
    perimeter={"airport_iata": "TIA", "month": 1},
    sources=SOURCES,
    user_query="Identify anomalous transit patterns at Tirana in January 2024",
)
final_state = orchestrator.invoke(initial)

print("=" * 70)
print("EXECUTION TRACE")
print("=" * 70)
for line in final_state["log"]:
    print(line)

if final_state.get("errors"):
    print("\nErrors:")
    for e in final_state["errors"]:
        print(" ", e)

print("\n" + "=" * 70)
print("FINAL REPORT (Markdown)")
print("=" * 70)
from IPython.display import Markdown
display(Markdown(final_state["report_md"]))

EXECUTION TRACE
[00:35:12] init | perimeter={'airport_iata': 'TIA', 'month': 1}
[data_agent] travelers 4987 -> 1216 | alarms 4986 -> 607
[baseline_agent] master shape=(1216, 50) | airports in subset=1
[outlier_agent] mode=ensemble | IF=37 LOF=37 Z=37
[risk_agent] confirmed_anomaly=31 | rules fired -> {'rate_3x': 70, 'traffic_2x': 12, 'volume_spike': 91}
[report_agent] LLM report generated (1629 chars)

FINAL REPORT (Markdown)


## Summary
The analysis covers transit patterns at Tirana International Airport (TIA) for January. A total of 1216 observations were processed within this perimeter. Of these records, 31 confirmed anomalies were identified by the system. The detected anomalies indicate periods of significantly elevated alarm rates and rule firings compared to established baselines.

## Top anomalies
*   **2024-01-07:** Multiple entries triggered high alarms (up to 20/entry) due to an alarm rate reaching 10.000–20.000 (baseline 0.639). This indicates severe, short-duration spikes in activity requiring immediate review of operational procedures.
*   **2024-01-11:** Multiple entries showed sustained high alarm rates (up to 20.000) and triggered rules related to rate increases and volume p99. Staff should investigate if staffing levels or processing bottlenecks contributed to these spikes.
*   **2024-01-03:** Two entries recorded a significant alarm rate of 9.500 (baseline 0.639) based on the rule [rate>=3x]. This suggests an unusual, sustained increase in activity that warrants review for systemic causes.

## Recommended next steps
*   Review operational logs for 2024-01-07 and 2024-01-11 to determine if staffing changes or procedural updates correlate with the observed high alarm rates (up to 20.000).
*   Cross-reference the dates of confirmed anomalies with known flight schedules or major event data for January to identify external triggers.
*   Adjust baseline thresholds for rate and volume p99 rules, particularly if the current spikes are determined to be normal operational fluctuations rather than security incidents.

In [ ]:
# Full-dataset run -> must produce the same number of confirmed_anomaly as classical (63)
full_state = orchestrator.invoke(init_state(perimeter={}, sources=SOURCES,
                                            user_query="Full-dataset anomaly review"))
n_agents    = int(full_state["flagged"]["confirmed_anomaly"].sum())
n_classical = int(df_master["confirmed_anomaly"].sum())
print(f"Agentic pipeline (full dataset): {n_agents} confirmed anomalies")
print(f"Classical pipeline:              {n_classical} confirmed anomalies")
print(f"Match: {'✅' if n_agents == n_classical else '⚠️'}")

Agentic pipeline (full dataset): 63 confirmed anomalies
Classical pipeline:              63 confirmed anomalies
Match: ✅


## 22. Quantitative comparison: classical vs multi-agent

We compare the two implementations on three orthogonal dimensions:

1. **Output identity** — do they produce the same anomalies on the same input?
2. **Latency** — how much does the multi-agent overhead cost in wall-clock time, both with and without the LLM call?
3. **Operational flexibility** — what queries can each implementation answer?

The first two are measurable; the third is structural and we summarise it as a feature matrix. Together they support the claim of section 16: the agentic system is not algorithmically *better*, it is operationally *different*.

In [ ]:
import time

# --- (1) Output identity (we already validated this in section 21.3, but we keep
#         it here as a single block so section 22 is self-contained for the pitch) ---
classical_confirmed = set(
    df_master.loc[df_master["confirmed_anomaly"] == 1].index.tolist()
)
agentic_full = orchestrator.invoke(init_state(perimeter={}, sources=SOURCES))
agentic_confirmed = set(
    agentic_full["flagged"].loc[agentic_full["flagged"]["confirmed_anomaly"] == 1].index.tolist()
)

intersection = classical_confirmed & agentic_confirmed
union        = classical_confirmed | agentic_confirmed
jaccard      = len(intersection) / len(union) if union else 0.0

print("=" * 70)
print("(1) OUTPUT IDENTITY — full dataset, no perimeter")
print("=" * 70)
print(f"  Classical confirmed anomalies: {len(classical_confirmed)}")
print(f"  Agentic   confirmed anomalies: {len(agentic_confirmed)}")
print(f"  Jaccard agreement:             {jaccard:.2%}")
print(f"  Verdict: {'identical sets ✅' if jaccard == 1.0 else 'sets diverge ⚠️'}")

(1) OUTPUT IDENTITY — full dataset, no perimeter
  Classical confirmed anomalies: 63
  Agentic   confirmed anomalies: 63
  Jaccard agreement:             100.00%
  Verdict: identical sets ✅


In [ ]:
# --- (2) Latency ---
N_RUNS = 3   # we average a few runs to smooth out cold-start effects

def time_classical_full() -> float:
    """Re-run the full classical pipeline from cleaned tables to confirmed_anomaly."""
    t0 = time.perf_counter()
    m = engineer_features(df_trav, df_alar)
    m = fit_detectors(m)
    _ = apply_business_rules(m)
    return time.perf_counter() - t0


def time_agentic_no_llm() -> float:
    """Agentic pipeline up to risk_agent only (skip the LLM call)."""
    t0 = time.perf_counter()
    s = init_state(perimeter={}, sources=SOURCES)
    s.update(data_agent(s))
    s.update(baseline_agent(s))
    s.update(outlier_agent(s))
    s.update(risk_agent(s))
    return time.perf_counter() - t0


def time_agentic_full() -> float:
    """Agentic pipeline including the Report Agent (LLM call)."""
    t0 = time.perf_counter()
    _ = orchestrator.invoke(init_state(perimeter={}, sources=SOURCES))
    return time.perf_counter() - t0


def avg(fn, n=N_RUNS) -> float:
    return float(np.mean([fn() for _ in range(n)]))


print("Timing (averages over", N_RUNS, "runs)...")
t_classical = avg(time_classical_full)
t_agent_no  = avg(time_agentic_no_llm)
t_agent_yes = avg(time_agentic_full)

print("\n" + "=" * 70)
print("(2) LATENCY — full dataset")
print("=" * 70)
print(f"  Classical pipeline:                {t_classical*1000:7.1f} ms")
print(f"  Agentic pipeline (no LLM):         {t_agent_no*1000:7.1f} ms  "
      f"(+{(t_agent_no - t_classical)*1000:.1f} ms overhead)")
print(f"  Agentic pipeline (with Gemma):     {t_agent_yes*1000:7.1f} ms  "
      f"({t_agent_yes/t_classical:.1f}x classical)")
print(f"\n  Of which LLM-only: ~{(t_agent_yes - t_agent_no)*1000:.0f} ms "
      f"({(t_agent_yes - t_agent_no)/t_agent_yes:.0%} of total)")

Timing (averages over 3 runs)...

(2) LATENCY — full dataset
  Classical pipeline:                  270.9 ms
  Agentic pipeline (no LLM):           225.3 ms  (+-45.6 ms overhead)
  Agentic pipeline (with Gemma):     38224.8 ms  (141.1x classical)

  Of which LLM-only: ~37999 ms (99% of total)


In [ ]:
# --- (3) Operational flexibility — feature matrix ---
flex_matrix = pd.DataFrame(
    [
        ["User-defined perimeter (single airport / month / country)",       "Manual re-run", "Native"],
        ["Per-perimeter baseline recomputation",                            "Manual re-run", "Native"],
        ["Natural-language report with operational recommendations",        "Not available", "Native (LLM)"],
        ["Reproducibility (deterministic numeric output)",                  "Native",        "Native"],
        ["Auditability of the analytical chain",                            "Linear cells",  "Per-agent log"],
        ["Robustness to small subsets",                                     "Single config", "Size-aware fallback"],
        ["External infrastructure dependency",                              "None",          "Local LLM server"],
        ["Latency on the full dataset",                                     "Fastest",       "~LLM call slower"],
    ],
    columns=["Capability", "Classical", "Multi-agent"],
)
display(flex_matrix)

,Capability,Classical,Multi-agent
0,User-defined perimeter (single airport / month...,Manual re-run,Native
1,Per-perimeter baseline recomputation,Manual re-run,Native
2,Natural-language report with operational recom...,Not available,Native (LLM)
3,Reproducibility (deterministic numeric output),Native,Native
4,Auditability of the analytical chain,Linear cells,Per-agent log
5,Robustness to small subsets,Single config,Size-aware fallback
6,External infrastructure dependency,None,Local LLM server
7,Latency on the full dataset,Fastest,~LLM call slower


### 22.1 Multi-perimeter robustness check

To support the claim that the agentic implementation handles arbitrary perimeters without manual reconfiguration, we run the same orchestrator on three different scopes and verify that each produces a coherent report. This is the *operational* test that the classical pipeline cannot pass without manual re-execution.

In [ ]:
PERIMETERS_TO_TEST = [
    {"airport_iata": "TIA"},                         # busiest airport, full window
    {"airport_iata": "STN", "month": 2},             # secondary airport, single month
    {"month": 1, "year": 2024},                      # all airports, single month
]

rows = []
for p in PERIMETERS_TO_TEST:
    s = orchestrator.invoke(init_state(perimeter=p, sources=SOURCES))
    flagged = s.get("flagged")
    rows.append({
        "perimeter":          str(p),
        "rows_in_scope":      0 if flagged is None else len(flagged),
        "confirmed_anomaly":  0 if flagged is None else int(flagged["confirmed_anomaly"].sum()),
        "report_chars":       len(s.get("report_md", "")),
        "errors":             len(s.get("errors", [])),
    })

multi_perim_summary = pd.DataFrame(rows)
display(multi_perim_summary)

print("\nAll runs completed without errors:" ,
      bool((multi_perim_summary["errors"] == 0).all()))

,perimeter,rows_in_scope,confirmed_anomaly,report_chars,errors
0,{'airport_iata': 'TIA'},2213,59,1700,0
1,"{'airport_iata': 'STN', 'month': 2}",247,5,1795,0
2,"{'month': 1, 'year': 2024}",2507,39,1553,0



All runs completed without errors: True


## 23. Conclusions — when to use which approach

This section consolidates the take-away from the comparative analysis required by the company brief. It is intentionally written as the discussion paragraph that ships with the README and the pitch deck.

### 23.1 What we built

We implemented the same anomaly-detection system twice on the Whitehall Reply transit dataset:

- A **classical pipeline** (sections 1–14) that joins the cleaned `travelers` and `alarms` tables, engineers leak-free features (per-airport rate baselines, strictly-past rolling means, traffic multipliers), runs a three-detector ensemble (Isolation Forest, Local Outlier Factor, Z-score), and applies a three-rule post-processing layer to surface *confirmed anomalies*.
- A **multi-agent pipeline** (sections 16–22) that decomposes the same logic into five specialist agents — Data, Baseline, Outlier Detection, Risk Profiling, Report — coordinated by a LangGraph supervisor, with a local Gemma 3n E4B LLM (served by LM Studio) invoked only by the Report Agent. The four analytical agents reuse the exact same Python helpers as the classical pipeline; on the full dataset, the two implementations produce **identical sets of confirmed anomalies** (Jaccard = 100%, section 22).

### 23.2 What the comparison teaches

The numbers in section 22 settle three questions that we had open at the start of the project:

1. **Does the agentic system change the detection quality?** No. The two pipelines flag the same events on the same data because they share the same math. The agentic system is a *re-decomposition* of the classical one, not a different algorithm.
2. **What does the agentic system cost?** The pure analytical overhead (Data → Risk, no LLM) is in the order of a few milliseconds — negligible compared to the classical pipeline. The dominant cost is the LLM call in the Report Agent, which adds the bulk of the wall-clock time. This is the price for human-readable reports.
3. **What does the agentic system give back for that cost?** Three things: (i) **per-perimeter baselines**, recomputed inside the user's scope, so deviations are local rather than global; (ii) **natural-language reports** with actionable recommendations, suited for an operator audience that does not read pandas tables; (iii) **a modular execution graph** that can be extended (new agent for hot-list cross-checks, new agent for SQL persistence) without touching the analytical core.

### 23.3 When to choose which

- **Choose the classical pipeline** when the workload is a single offline batch on a known, stable dataset, when latency and reproducibility are paramount, and when the consumer of the output is technical (data analyst, ML engineer). It is faster, simpler, has zero external dependencies, and is the right baseline against which any agentic version should be benchmarked — exactly the role it played in this project.
- **Choose the multi-agent pipeline** when the analysis is *interactive and perimeter-driven* (the operator wants Tirana in January, then Stansted in February, then "all flights from Country X last week"), when the consumer is operational and benefits from a written report, or when the system is expected to grow in scope over time. The agentic decomposition is what makes such growth tractable: each new capability is a new node in the graph, not a refactoring of a monolithic notebook.

### 23.4 Limitations and natural next steps

- **Observation window.** The dataset covers ~3 months (Dec 2023 → Feb 2024), which is too short for seasonal decomposition; we used rolling means as the only temporal baseline. With a 12+ month window, an explicit seasonal component could be added inside the Baseline Agent without changing any other part of the graph.
- **Ground truth.** We have no labelled anomalies, so we could not compute precision and recall. The Jaccard agreement between detectors and the rule-based confirmation are our best proxies. A natural next step is to back-test the surfaced events against operator labels for a future window.
- **Small open-weight model.** Gemma 3n E4B was sufficient for the Report Agent because we hand-fed it a pre-digested context block; we deliberately kept analytical reasoning out of the LLM. A larger model (e.g. Qwen 2.5 7B Instruct) would allow more sophisticated tasks at the LLM level — for instance, a future *Perimeter Parsing Agent* that turns free-form user queries ("anomalies last week at major hubs") into the structured perimeter dictionary that the Data Agent currently expects.
- **Streamlit UI.** The Whitehall Reply brief suggests a simple front-end to expose the perimeter selection and the report. The orchestrator is already a `Runnable` with a clean state interface, so the UI is a thin wrapper around `orchestrator.invoke(...)` — out of scope for this notebook but a natural deliverable for the pitch.